# Credit Card Fraud Detection — Proof of Concept

## Executive Summary

This document presents a **proof-of-concept fraud detection system** built for our financial services organization. The system uses machine learning to automatically scan credit card transactions and flag those that appear fraudulent, enabling our fraud operations team to focus their review efforts where it matters most.

**The business problem:** Credit card fraud costs the global financial industry tens of billions of dollars every year. Manual review of every transaction is impossible — our dataset alone contains nearly 285,000 transactions. We need an automated first line of defense that can rapidly identify suspicious activity.

**Our approach:** We use Microsoft Azure's cloud-based machine learning platform to build, train, and deploy a model that learns what "normal" transactions look like and then flags transactions that deviate from that pattern. This technique is called *anomaly detection*.

**Key finding:** Our initial model correctly identifies normal transactions with near-perfect accuracy, and detects approximately 28% of actual fraud cases. While this is a meaningful starting point, it also highlights the need for iterative improvement before production deployment — a normal and expected outcome for a first proof of concept.

---

## How the Pipeline Works

The diagram below shows the end-to-end process from raw data to a deployed, usable model:

```
┌─────────────────────────────────────────────────────────────────────┐
│                    FRAUD DETECTION ML PIPELINE                      │
└─────────────────────────────────────────────────────────────────────┘

  ┌───────────────┐    ┌───────────────┐    ┌──────────────────────┐
  │  1. DATA      │    │  2. DATA      │    │  3. MODEL            │
  │  INGESTION    │───>│  PREPARATION  │───>│  TRAINING            │
  │               │    │               │    │                      │
  │ Load credit   │    │ Standardize   │    │ Isolation Forest     │
  │ card records  │    │ dollar amounts│    │ learns "normal"      │
  │ from Azure    │    │ Remove unused │    │ transaction patterns │
  └───────────────┘    │ fields        │    └──────────┬───────────┘
                       └───────────────┘               │
                                                       ▼
  ┌───────────────┐    ┌───────────────┐    ┌──────────────────────┐
  │  6. BUSINESS  │    │  5. VISUAL    │    │  4. MODEL            │
  │  DECISION     │<───│  ANALYSIS     │<───│  EVALUATION          │
  │               │    │               │    │                      │
  │ Review results│    │ Charts &      │    │ Measure accuracy,    │
  │ Refine model  │    │ explainability│    │ precision, & recall  │
  │ Plan rollout  │    │ dashboards    │    │ on known fraud cases │
  └───────────────┘    └───────────────┘    └──────────────────────┘
```

---

## Azure ML Platform Components

The table below summarizes the key Microsoft Azure components used in this solution and the role each plays:

| Azure ML Component | Role in Our Fraud Detection Pipeline |
|---|---|
| **Azure ML Workspace** | The central hub that organizes all project resources — datasets, models, experiments, and compute — in one secure location. |
| **Azure ML Datasets** | Stores and versions our credit card transaction data so it can be reliably accessed by any team member or automated process. |
| **Azure ML Notebooks** | Provides the interactive environment where our data scientists build, test, and document the model step by step. |
| **Azure ML Compute** | Supplies the cloud processing power needed to train the model on nearly 285,000 transactions without requiring local hardware. |
| **Azure ML Model Registry** | Stores the trained model with version tracking, so we can compare improvements over time and roll back if needed. |
| **Azure ML Endpoints** | Enables the trained model to be deployed as a live service that our transaction processing systems can call in real time. |
| **Azure CLI / SDK** | The programmatic interface that allows our technical team to automate and reproduce every step of the pipeline. |

---

## Workflow

The sections below walk through each step of the pipeline, explaining what happens and why it matters from a business perspective.

### Step 1: Load Required Software Components

Before any analysis can begin, the system loads the specialized software libraries it depends on. Think of this like opening the specific applications you need before starting a task on your computer.

The components loaded here provide four capabilities:
- **Connection to Azure** — securely links this notebook to our cloud workspace where the data and models are stored.
- **Data handling** — gives us tools to organize and manipulate the transaction data in table form.
- **Machine learning algorithm** — loads the Isolation Forest model, which is the core engine that will learn to distinguish normal transactions from suspicious ones.
- **Performance measurement** — provides standardized tools for measuring how well the model performs.

In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

### Step 2: Connect to Azure and Load the Transaction Data

This step securely connects to our Azure ML cloud workspace and retrieves the credit card transaction dataset that was previously uploaded and registered there.

**What is the dataset?** It contains 284,807 real credit card transactions from European cardholders, collected over two days. Of these, only 492 (0.17%) are confirmed fraud — which reflects the real-world challenge that fraudulent transactions are extremely rare compared to legitimate ones.

**Why use Azure for this?** Storing the data in Azure ensures that every team member works from the same authoritative dataset, that changes are tracked, and that the data is protected by enterprise-grade security. It also means we are not dependent on any single person's laptop or local files.

The code below connects to the workspace, pulls the dataset, and displays a preview of the first few rows to confirm everything loaded correctly.

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())


In [ ]:
# if you're running locally then use this ...
path = None

# alternatively, if you're running in Azure AI Machine Learning Studio - Notebook, then use this ...
# (make sure to upload the config.json file to the same directory as this notebook)
#  and then execute this code to determine the current working directory.
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()

### Step 3: Prepare the Data for Analysis

Before the model can analyze the transactions, we need to prepare the data. This step performs two important adjustments:

1. **Standardize the transaction dollar amounts.** The raw dollar values range widely (from less than a dollar to thousands). The model works more effectively when all numeric values are on a comparable scale, so we convert the dollar amounts to a standardized range. This is similar to how you might convert different currencies to a single baseline for comparison — the actual transaction amounts are preserved, just re-expressed.

2. **Remove irrelevant information.** The dataset includes a timestamp field that records when each transaction occurred. Since our model focuses on transaction *characteristics* rather than timing, we remove this field so it does not introduce noise into the analysis.

After this step, the data is cleanly organized into two parts: the **transaction features** (the information the model will analyze) and the **fraud labels** (the known answers we will use to measure the model's accuracy).

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Train the Fraud Detection Model

This is the core step where the system learns to identify suspicious transactions. We use an algorithm called **Isolation Forest**, which is specifically designed for anomaly detection — finding the "needles in the haystack."

**How does it work in plain terms?** Imagine sorting a deck of cards by asking yes/no questions. Normal cards, which all look similar, take many questions to tell apart. But a joker — the outlier — can be identified in just one or two questions because it is obviously different. Isolation Forest applies this same logic at massive scale across all 284,807 transactions.

**Key configuration:** We tell the model to expect that approximately 0.17% of transactions are fraudulent, which matches the known fraud rate in our dataset. This calibration helps the model make appropriately conservative predictions rather than flagging too many or too few transactions.

After training, the model scores every transaction as either **normal** or **suspected fraud**.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Evaluate Model Performance

Now we measure how well the model performed by comparing its predictions against the known answers. This is the most important step for decision-making because it tells us where the model succeeds and where it falls short.

**Key metrics explained:**

| Metric | What It Tells Us | Our Result |
|---|---|---|
| **Precision** (for fraud) | When the model flags a transaction as fraud, how often is it actually fraud? | **29%** — roughly 1 in 3 flagged transactions is real fraud |
| **Recall** (for fraud) | Of all the real fraud in the dataset, how much did the model catch? | **28%** — the model catches about 1 in 4 fraud cases |
| **Overall Accuracy** | What percentage of all transactions were classified correctly? | **99.9%** — but see the important caveat below |

**Important caveat about accuracy:** The 99.9% overall accuracy is misleading on its own. Because fraud represents only 0.17% of all transactions, a model that simply labels *everything* as "normal" would still be 99.83% accurate while catching zero fraud. This is why **precision** and **recall** are the metrics that matter most for evaluating a fraud detection system.

**What this means for the business:** The model in its current form is a useful starting point — it catches roughly one-quarter of fraud cases and could be used as a screening layer that routes flagged transactions to human reviewers. However, it would miss approximately three-quarters of actual fraud and would generate a significant number of false alarms. These are expected results for an initial proof of concept and establish a clear baseline for improvement.

In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Step 6: Save and Register the Trained Model in Azure

Once we are satisfied with a version of the model, we save it to Azure's **Model Registry**. This is similar to checking a document into a version-controlled filing system — it creates a permanent, retrievable record of this specific model.

**Why this matters:**
- **Reproducibility** — anyone on the team can retrieve this exact model months from now and get identical results.
- **Version tracking** — as we improve the model over time, each version is stored separately so we can compare performance and roll back if a new version underperforms.
- **Deployment readiness** — a registered model can be deployed as a live service (an API endpoint) that our transaction processing systems call in real time to score new transactions as they occur.

In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Visualize the Results

Visualizations give us an intuitive, at-a-glance understanding of how the model behaves. The charts below serve as a "dashboard view" of the model's output.

#### Chart 1: Count of Predicted Anomalies

The bar chart below shows the total number of transactions the model classified as **normal (0)** versus **suspected fraud (1)**.

**What to look for:** You should see a very tall bar for normal transactions and a very short bar for suspected fraud. This reflects the reality that fraud is rare. If the number of flagged transactions is close to the ~492 known fraud cases in the dataset, the model is appropriately calibrated. If it flags far more, the model may be too aggressive (creating excessive false alarms); if far fewer, it may be too conservative (missing fraud).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


#### Chart 2: Transaction Amounts — Normal vs. Flagged Transactions

The box plot below compares the dollar amounts of transactions the model predicted as normal versus those it flagged as suspicious.

**What to look for:** If flagged transactions tend to have more extreme or widely varying dollar amounts, it suggests the model is treating unusual transaction sizes as a signal of potential fraud. This is a reasonable behavior, but it also reveals a limitation — fraud can occur at any dollar amount, and a model that only flags large transactions would miss small-dollar fraud schemes.

This chart helps us understand whether the model has any blind spots related to transaction size that we should address in future iterations.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


#### Chart 3: Feature Importance — What Drives the Model's Decisions?

The chart below (called a SHAP beeswarm plot) answers a critical question for stakeholders: **why** did the model flag specific transactions?

Each row represents one characteristic (feature) of a transaction. Each dot is an individual transaction. Dots pushed to the right indicate that feature pushed the model toward a "fraud" prediction; dots pushed to the left indicate it pushed toward "normal." The color indicates whether the feature's value was high (red) or low (blue) for that transaction.

**Why this matters for governance and trust:** In financial services, regulators and auditors increasingly require that automated decision-making systems be *explainable*. This chart demonstrates that our model's decisions can be broken down and understood — we can identify which transaction characteristics most influence a fraud flag, which is essential for regulatory compliance, internal audit, and customer dispute resolution.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)

---

## Business Impact Assessment

### Cost-Benefit Analysis: False Positives vs. Missed Fraud

Every fraud detection system must balance two types of errors, each with distinct business costs:

| Error Type | What Happens | Business Impact | Our Model's Tendency |
|---|---|---|---|
| **False Positive** (false alarm) | A legitimate transaction is flagged as fraud | Customer friction, blocked purchases, call center volume, potential customer churn | ~71% of the model's fraud flags are false alarms |
| **False Negative** (missed fraud) | A real fraud transaction is not caught | Direct financial loss, regulatory liability, customer trust damage, chargeback costs | ~72% of actual fraud cases are missed |

**The tradeoff:** Tightening the model to catch more fraud will inevitably increase false alarms, and vice versa. The right balance depends on our organization's risk appetite, the average dollar value of fraud vs. the cost of investigating false alarms, and customer experience priorities. Based on industry benchmarks, the cost of a single missed fraud event typically outweighs the cost of investigating several false alarms, which suggests we should prioritize improving recall in future model iterations.

---

### Recommendations for Model Improvement and Deployment

This proof of concept establishes a working baseline. The following improvements are recommended before production deployment:

1. **Incorporate supervised learning techniques.** Our current model (Isolation Forest) is *unsupervised* — it does not directly learn from the known fraud labels. Adding a supervised model (such as XGBoost or a neural network) that trains on labeled examples would likely improve fraud detection rates significantly.

2. **Use ensemble methods.** Combining multiple models — for example, running Isolation Forest alongside a supervised classifier and requiring agreement — can reduce both false positives and missed fraud.

3. **Engineer additional features.** Transaction velocity (how many transactions in the last hour), geographic distance from the cardholder's home, merchant category risk scores, and device fingerprinting can all strengthen detection.

4. **Implement threshold tuning.** Rather than a binary "fraud or not" decision, deploy a risk-scoring approach where transactions above a high threshold are auto-blocked, those in a middle range are routed to human review, and those below are approved — reducing both customer friction and fraud losses.

5. **Establish a feedback loop.** Connect confirmed fraud cases and false alarm resolutions back into the training data so the model continuously improves over time.

6. **Conduct A/B testing before full rollout.** Deploy the model on a subset of transactions in parallel with existing processes to validate real-world performance before replacing current systems.

---

### Risk Assessment and Mitigation

| Risk | Likelihood | Impact | Mitigation Strategy |
|---|---|---|---|
| Model misses a high-value fraud event | High (given current 28% recall) | High — direct financial loss | Layer model with rule-based checks for high-value transactions; maintain human review for transactions above a dollar threshold |
| Excessive false positives frustrate customers | Medium | Medium — customer churn, brand damage | Implement tiered scoring rather than binary blocking; optimize the decision threshold based on cost analysis |
| Model performance degrades over time as fraud patterns evolve | High | High — increasing losses | Schedule quarterly model retraining; monitor detection rates continuously with automated alerting |
| Regulatory or audit concerns about automated decisions | Medium | High — compliance violations, fines | Maintain SHAP-based explainability (demonstrated in Step 7); document all model decisions; establish a human appeals process |
| Data privacy breach of transaction records | Low | Very High — regulatory fines, lawsuits | Leverage Azure's enterprise security, encryption at rest and in transit, role-based access controls |

---

### Stakeholder Communication Plan for Model Limitations

Transparent communication about what this model can and cannot do is essential for setting appropriate expectations:

**For executive leadership:** This proof of concept validates that machine learning can be applied to our transaction data to detect fraud patterns automatically. The initial model catches roughly one in four fraud cases — a meaningful starting point that demonstrates feasibility. We recommend a phased improvement roadmap with clear performance targets at each stage before full production deployment.

**For the fraud operations team:** The model will serve as an additional screening tool, not a replacement for existing processes. Flagged transactions should be treated as leads for investigation, not confirmed fraud. We expect the false alarm rate to decrease as the model is refined with your team's feedback.

**For customer-facing teams:** In its current form, the model will not directly block transactions or impact customers. If deployed in a later phase, a tiered approach will be used to minimize disruption to legitimate transactions, and customers will always have access to a rapid resolution process.

**For compliance and legal:** The model uses an explainable framework (SHAP) that can provide feature-level justification for any individual decision. All model versions, training data, and performance metrics are versioned and stored in Azure's Model Registry for audit purposes.

*Created with assistance form ClaudeAI*